# RLHF with Hugging Face TRL

This notebook is the restartable Colab control plane for the Qwen2.5-0.5B HelpSteer3 RLHF pipeline. The repository can live on Google Drive, while training outputs are written to the fast Colab SSD and synchronized back to Drive after checkpoints and final artifacts.

The stages are independent on purpose: after a runtime disconnect, rerun setup and restore cells, then jump directly to the next unfinished section.

## 0. Mount Drive and choose paths

Update `REPO_ROOT` to the Drive checkout you already pulled with your separate clone/update notebook. No git clone happens here.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import shlex
import shutil
import subprocess
import sys

REPO_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/8mile/rlhf")
DRIVE_SYNC_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/8mile/rlhf_runs")
DRIVE_LIGHTWEIGHT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/8mile/rlhf_runs_lightweight_export")

RUN_TAG = "qwen25_05b_helpsteer3_trl_a100_full"
RUN_PROFILE = "full"  # "smoke" or "full"
RUN_FULL_EVAL = True

assert RUN_PROFILE in {"smoke", "full"}
if not (REPO_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(f"REPO_ROOT does not look like the RLHF repository: {REPO_ROOT}")

os.chdir(REPO_ROOT)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

LOCAL_RUN_ROOT = Path("/content/rlhf_runs") / RUN_TAG / RUN_PROFILE
DRIVE_RUN_ROOT = DRIVE_SYNC_ROOT / RUN_TAG / RUN_PROFILE
LIGHTWEIGHT_RUN_ROOT = DRIVE_LIGHTWEIGHT_ROOT / RUN_TAG / RUN_PROFILE

CACHE_DIR = LOCAL_RUN_ROOT / "data"
SFT_DIR = LOCAL_RUN_ROOT / "sft"
REWARD_DIR = LOCAL_RUN_ROOT / "reward"
REWARD_EPOCH2_DIR = LOCAL_RUN_ROOT / "reward_epoch2"

PPO_RESPONSE_LENGTH = 768
PPO_KL_COEF = 0.07
PPO_RUN_NAME = f"ppo_nplus_exact_eos_r{PPO_RESPONSE_LENGTH}_b64_kl07_rm_ep2"
PPO_DIR = LOCAL_RUN_ROOT / PPO_RUN_NAME
EVAL_DIR = LOCAL_RUN_ROOT / f"eval_{PPO_RUN_NAME}_latest_full"

for directory in [LOCAL_RUN_ROOT, CACHE_DIR, SFT_DIR, REWARD_DIR, REWARD_EPOCH2_DIR, PPO_DIR, EVAL_DIR, DRIVE_RUN_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

SMOKE = RUN_PROFILE == "smoke"
print("Repository root:", REPO_ROOT)
print("Local run root:", LOCAL_RUN_ROOT)
print("Drive run root:", DRIVE_RUN_ROOT)
print("PPO run name:", PPO_RUN_NAME)

## 1. Install and verify the runtime

Use normal notebook `%pip` commands for installation. The scripts themselves are run as subprocesses later so their logs stream cleanly and failures show the child-process tail.

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

In [ ]:
import accelerate
import datasets
import peft
import torch
import transformers
import trl

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)

from trl import RewardConfig, RewardTrainer, SFTConfig, SFTTrainer
from trl.experimental.ppo import PPOConfig, PPOTrainer

print("TRL trainer APIs available:", SFTTrainer.__name__, RewardTrainer.__name__, PPOTrainer.__name__)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before training.")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("GPU memory (GiB):", round(props.total_memory / 2**30, 1))
torch.backends.cuda.matmul.allow_tf32 = True

## 2. Shared helpers

These helpers make the notebook restartable. Restore a stage from Drive when the local runtime is fresh, then run only the unfinished stage.

In [ ]:
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TQDM_MININTERVAL", "5")


def encode_override(key, value):
    if isinstance(value, (str, bool, int, float)) or value is None:
        rendered = json.dumps(value)
    else:
        rendered = json.dumps(value, separators=(",", ":"))
    return f"{key}={rendered}"


def run_script(script, config, overrides=None, extra_args=None):
    command = [sys.executable, script, "--config", str(config)]
    for key, value in (overrides or {}).items():
        command.extend(["--set", encode_override(key, value)])
    command.extend(extra_args or [])
    print("$", shlex.join(command))
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    env["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        tail.append(line)
        tail = tail[-120:]
    return_code = process.wait()
    if return_code:
        detail = "".join(tail)
        message = (
            f"Command failed with exit status {return_code}: {shlex.join(command)}\n\n"
            f"Last child-process output:\n{detail}"
        )
        raise RuntimeError(message)


def copy_tree(source, destination):
    source = Path(source)
    destination = Path(destination)
    if not source.exists():
        print("Nothing to copy:", source)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print("Copied:", source, "->", destination)


def sync_overrides(stage):
    stage_root = DRIVE_RUN_ROOT / stage
    return {
        "train.checkpoint_sync_dir": str(stage_root / "checkpoints"),
        "train.final_sync_dir": str(stage_root),
    }


def restore_stage(stage, local_dir):
    source = DRIVE_RUN_ROOT / stage
    local_dir = Path(local_dir)
    if not source.exists():
        print("No Drive stage to restore:", source)
        return False
    local_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, local_dir, dirs_exist_ok=True)
    print("Restored:", source, "->", local_dir)
    return True


def stage_complete(path, required):
    path = Path(path)
    return all((path / name).exists() for name in required)


def checkpoint_step(path):
    match = re.fullmatch(r"checkpoint-(\d+)", Path(path).name)
    return int(match.group(1)) if match else -1


def latest_trainer_checkpoint(*roots):
    candidates = []
    for root in roots:
        if root is None:
            continue
        root = Path(root)
        if not root.exists():
            continue
        for path in root.rglob("checkpoint-*"):
            if path.is_dir() and checkpoint_step(path) >= 0:
                if (path / "trainer_state.json").exists() or (path / "adapter_config.json").exists():
                    candidates.append(path)
    if not candidates:
        return None
    return sorted(candidates, key=lambda path: (checkpoint_step(path), len(path.parts)))[-1]


def find_policy_adapter_dir(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    candidates = []
    for config_path in checkpoint_dir.rglob("adapter_config.json"):
        try:
            payload = json.loads(config_path.read_text())
        except Exception:
            payload = {}
        if payload.get("task_type") == "CAUSAL_LM":
            candidates.append(config_path.parent)
    if not candidates:
        candidates = [path.parent for path in checkpoint_dir.rglob("adapter_config.json")]
    if not candidates:
        print("Checkpoint contents:")
        for path in sorted(checkpoint_dir.rglob("*"))[:200]:
            print(" ", path)
        raise FileNotFoundError(f"No adapter_config.json found under {checkpoint_dir}")
    return sorted(candidates, key=lambda path: len(path.parts))[0]


def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
for stage, local_dir in [
    ("data", CACHE_DIR),
    ("sft", SFT_DIR),
    ("reward", REWARD_DIR),
    ("reward_epoch2", REWARD_EPOCH2_DIR),
    (PPO_RUN_NAME, PPO_DIR),
]:
    restore_stage(stage, local_dir)

ACTIVE_REWARD_DIR = REWARD_EPOCH2_DIR if stage_complete(REWARD_EPOCH2_DIR, ["final_merged_model", "reward_center.json"]) else REWARD_DIR
print("Active reward model:", ACTIVE_REWARD_DIR)

## 3. Prepare response-safe datasets

This creates SFT, reward, and PPO-ready HelpSteer3 datasets under the local run root, then copies them to Drive for restart safety.

In [ ]:
RUN_PREPARE = True
FORCE_PREPARE = False

prepare_overrides = {
    "data.cache_dir": str(CACHE_DIR),
}
if SMOKE:
    prepare_overrides.update({
        "data.max_train_samples": 256,
        "data.max_validation_samples": 64,
        "data.max_total_length": 1024,
        "data.max_prompt_length": 768,
    })

if stage_complete(CACHE_DIR, ["preparation_report.json"]) and not FORCE_PREPARE:
    print("Prepared data already exists:", CACHE_DIR)
elif RUN_PREPARE:
    run_script(
        "scripts/rlhf_trl_prepare_data.py",
        "configs/trl/qwen25_05b_helpsteer3_sft.yaml",
        prepare_overrides,
    )
    copy_tree(CACHE_DIR, DRIVE_RUN_ROOT / "data")
else:
    print("Skipping data preparation.")

## 4. Supervised fine-tuning

Rerun setup and restore cells first if Colab reconnects. This section skips automatically when `final_merged_model` is present unless `FORCE_SFT=True`.

In [ ]:
RUN_SFT = True
FORCE_SFT = False

sft_overrides = {
    "data.cache_dir": str(CACHE_DIR),
    "train.output_dir": str(SFT_DIR),
    **sync_overrides("sft"),
}
if SMOKE:
    sft_overrides.update({
        "data.max_train_samples": 128,
        "data.max_eval_samples": 32,
        "train.max_steps": 8,
        "train.per_device_train_batch_size": 2,
        "train.gradient_accumulation_steps": 2,
        "train.eval_steps": 4,
        "train.save_steps": 4,
    })

if stage_complete(SFT_DIR, ["final_merged_model"]) and not FORCE_SFT:
    print("SFT already complete:", SFT_DIR)
elif RUN_SFT:
    run_script(
        "scripts/rlhf_trl_doctor.py",
        "configs/trl/qwen25_05b_helpsteer3_sft.yaml",
        sft_overrides,
        extra_args=["--stage", "sft"],
    )
    run_script("scripts/rlhf_trl_train_sft.py", "configs/trl/qwen25_05b_helpsteer3_sft.yaml", sft_overrides)
else:
    print("Skipping SFT.")
clear_gpu_cache()

## 5. Reward model, epoch 1

This trains the scalar reward model from the SFT checkpoint, writes a reward center, and syncs final artifacts plus checkpoints to Drive.

In [ ]:
RUN_REWARD = True
FORCE_REWARD = False

reward_overrides = {
    "model.sft_model_path": str(SFT_DIR / "final_merged_model"),
    "data.cache_dir": str(CACHE_DIR),
    "train.output_dir": str(REWARD_DIR),
    **sync_overrides("reward"),
}
if SMOKE:
    reward_overrides.update({
        "data.max_train_samples": 128,
        "data.max_eval_samples": 32,
        "data.max_center_samples": 64,
        "train.max_steps": 8,
        "train.per_device_train_batch_size": 2,
        "train.gradient_accumulation_steps": 2,
        "train.eval_steps": 4,
        "train.save_steps": 4,
        "train.audit_batch_size": 8,
    })

if stage_complete(REWARD_DIR, ["final_merged_model", "reward_center.json"]) and not FORCE_REWARD:
    print("Reward epoch 1 already complete:", REWARD_DIR)
elif RUN_REWARD:
    run_script(
        "scripts/rlhf_trl_doctor.py",
        "configs/trl/qwen25_05b_helpsteer3_reward.yaml",
        reward_overrides,
        extra_args=["--stage", "reward"],
    )
    run_script(
        "scripts/rlhf_trl_train_reward_model.py",
        "configs/trl/qwen25_05b_helpsteer3_reward.yaml",
        reward_overrides,
    )
else:
    print("Skipping reward epoch 1.")
clear_gpu_cache()

## 6. Optional reward model continuation

Use this for the two-epoch reward model. It first tries exact Trainer checkpoint resume. If only the merged model exists, it starts a fresh one-epoch continuation from that merged reward checkpoint without reinitializing the score head.

In [ ]:
RUN_REWARD_EPOCH2 = True
FORCE_REWARD_EPOCH2 = False

reward_epoch2_overrides = {
    "data.cache_dir": str(CACHE_DIR),
    "train.output_dir": str(REWARD_EPOCH2_DIR),
    "train.run_name": "qwen25-05b-helpsteer3-trl-reward-epoch2",
    "train.save_total_limit": 3,
    **sync_overrides("reward_epoch2"),
}

if stage_complete(REWARD_EPOCH2_DIR, ["final_merged_model", "reward_center.json"]) and not FORCE_REWARD_EPOCH2:
    print("Reward epoch 2 already complete:", REWARD_EPOCH2_DIR)
    ACTIVE_REWARD_DIR = REWARD_EPOCH2_DIR
elif RUN_REWARD_EPOCH2:
    resume_checkpoint = latest_trainer_checkpoint(REWARD_DIR, REWARD_DIR / "checkpoints", DRIVE_RUN_ROOT / "reward")
    if resume_checkpoint:
        print("Exact reward resume checkpoint:", resume_checkpoint)
        reward_epoch2_overrides.update({
            "model.sft_model_path": str(SFT_DIR / "final_merged_model"),
            "model.initialize_reward_head": True,
            "train.resume_from_checkpoint": str(resume_checkpoint),
            "train.num_train_epochs": 2,
        })
    else:
        print("No Trainer checkpoint found; continuing from merged reward model.")
        reward_epoch2_overrides.update({
            "model.sft_model_path": str(REWARD_DIR / "final_merged_model"),
            "model.initialize_reward_head": False,
            "train.resume_from_checkpoint": None,
            "train.num_train_epochs": 1,
        })
    run_script(
        "scripts/rlhf_trl_doctor.py",
        "configs/trl/qwen25_05b_helpsteer3_reward.yaml",
        reward_epoch2_overrides,
        extra_args=["--stage", "reward"],
    )
    run_script(
        "scripts/rlhf_trl_train_reward_model.py",
        "configs/trl/qwen25_05b_helpsteer3_reward.yaml",
        reward_epoch2_overrides,
    )
    ACTIVE_REWARD_DIR = REWARD_EPOCH2_DIR
else:
    print("Skipping reward epoch 2; using reward epoch 1.")
    ACTIVE_REWARD_DIR = REWARD_DIR

print("ACTIVE_REWARD_DIR:", ACTIVE_REWARD_DIR)
clear_gpu_cache()

## 7. PPO alignment

This is the final N+ style PPO setup: fixed-length generation, required EOS reward replacement, reward whitening, zero dropout in loaded models, controlled sampling defaults, RM-initialized critic, and the Adam epsilon used in the N+ implementation notes. Stop early whenever checkpoints and evaluation look good; checkpoint sync keeps Drive up to date.

In [ ]:
RUN_PPO = True
FORCE_PPO = False

ppo_overrides = {
    "model.policy_model_path": str(SFT_DIR / "final_merged_model"),
    "model.reference_model_path": str(SFT_DIR / "final_merged_model"),
    "model.reward_model_path": str(ACTIVE_REWARD_DIR / "final_merged_model"),
    "model.reward_center_path": str(ACTIVE_REWARD_DIR / "reward_center.json"),
    "model.value_model_path": str(ACTIVE_REWARD_DIR / "final_merged_model"),
    "data.cache_dir": str(CACHE_DIR),
    "train.output_dir": str(PPO_DIR),
    **sync_overrides(PPO_RUN_NAME),
}

ppo_overrides.update({
    "ppo.total_episodes": 12000,
    "ppo.response_length": PPO_RESPONSE_LENGTH,
    "ppo.temperature": 0.7,
    "ppo.kl_coef": PPO_KL_COEF,
    "ppo.num_ppo_epochs": 4,
    "ppo.num_mini_batches": 1,
    "ppo.cliprange": 0.2,
    "ppo.cliprange_value": 0.2,
    "ppo.vf_coef": 0.1,
    "ppo.gamma": 1.0,
    "ppo.lam": 0.95,
    "ppo.whiten_rewards": True,
    "ppo.stop_token": "eos",
    "ppo.fixed_length_generation": True,
    "ppo.require_eos_for_reward": True,
    "ppo.missing_eos_reward": -1.0,
    "ppo.missing_eos_penalty": 0.0,
    "ppo.local_rollout_forward_batch_size": 2,
    "ppo.num_sample_generations": 1,
    "train.per_device_train_batch_size": 2,
    "train.gradient_accumulation_steps": 32,
    "train.per_device_eval_batch_size": 2,
    "data.max_eval_samples": 64,
    "train.learning_rate": 3.0e-6,
    "train.adam_epsilon": 1.0e-5,
    "train.lr_scheduler_type": "linear",
    "train.warmup_ratio": 0.0,
    "train.save_steps": 50,
    "train.save_total_limit": 8,
    "train.logging_steps": 1,
})
if SMOKE:
    ppo_overrides.update({
        "data.max_train_samples": 64,
        "data.max_eval_samples": 16,
        "ppo.total_episodes": 32,
        "ppo.response_length": 128,
        "ppo.local_rollout_forward_batch_size": 2,
        "train.per_device_train_batch_size": 1,
        "train.gradient_accumulation_steps": 8,
        "train.save_steps": 2,
    })

if stage_complete(PPO_DIR, ["final_merged_policy"]) and not FORCE_PPO:
    print("PPO already complete:", PPO_DIR)
elif RUN_PPO:
    run_script(
        "scripts/rlhf_trl_doctor.py",
        "configs/trl/qwen25_05b_helpsteer3_ppo.yaml",
        ppo_overrides,
        extra_args=["--stage", "ppo"],
    )
    run_script("scripts/rlhf_trl_train_ppo.py", "configs/trl/qwen25_05b_helpsteer3_ppo.yaml", ppo_overrides)
else:
    print("Skipping PPO.")
clear_gpu_cache()

## 8. Evaluate the latest PPO checkpoint

This evaluates Base, SFT, and the latest PPO adapter checkpoint on the full validation set unless `RUN_FULL_EVAL=False` or `SMOKE=True`.

In [ ]:
from rlhf.config import load_config, save_config

latest_checkpoint = latest_trainer_checkpoint(PPO_DIR, PPO_DIR / "checkpoints", DRIVE_RUN_ROOT / PPO_RUN_NAME)
if latest_checkpoint is None:
    raise FileNotFoundError(f"No PPO checkpoint found under {PPO_DIR} or Drive sync path.")
latest_step = checkpoint_step(latest_checkpoint)
adapter_latest = find_policy_adapter_dir(latest_checkpoint)
ppo_eval_label = "ppo_trl"

print("Latest PPO checkpoint:", latest_checkpoint)
print("Latest PPO update step:", latest_step)
print("Policy adapter:", adapter_latest)

EVAL_DIR = LOCAL_RUN_ROOT / f"eval_{PPO_RUN_NAME}_step{latest_step}_full"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_config(REPO_ROOT / "configs/trl/qwen25_05b_helpsteer3_eval_suite.yaml")
cfg["model"]["tokenizer_path"] = str(SFT_DIR / "final_merged_model")
cfg["model"]["torch_dtype"] = "bfloat16"
cfg["model"]["local_files_only"] = False
cfg["reward_model"]["checkpoint_dir"] = str(ACTIVE_REWARD_DIR / "final_merged_model")
cfg["reward_model"]["reward_center_path"] = str(ACTIVE_REWARD_DIR / "reward_center.json")
cfg["reward_model"]["torch_dtype"] = "bfloat16"
cfg["policies"] = [
    {"label": "base", "format": "trl", "checkpoint_dir": None, "torch_dtype": "bfloat16"},
    {"label": "sft_trl", "format": "trl", "checkpoint_dir": str(SFT_DIR / "final_merged_model"), "torch_dtype": "bfloat16"},
    {
        "label": ppo_eval_label,
        "format": "peft",
        "checkpoint_dir": str(SFT_DIR / "final_merged_model"),
        "base_model_path": str(SFT_DIR / "final_merged_model"),
        "adapter_path": str(adapter_latest),
        "torch_dtype": "bfloat16",
    },
]
cfg["generation"]["max_prompt_length"] = 3072
cfg["generation"]["max_new_tokens"] = 1024
cfg["generation"]["temperature"] = 0.7
cfg["generation"]["do_sample"] = True
cfg["eval"]["output_dir"] = str(EVAL_DIR)
cfg["eval"]["num_prompts"] = 32 if SMOKE else ("all" if RUN_FULL_EVAL else 256)
cfg["eval"]["batch_size"] = 4 if SMOKE else 256
cfg["eval"]["resume"] = True
cfg["eval"]["progress_every"] = 5

TEMP_EVAL_CONFIG = PPO_DIR / f"eval_{ppo_eval_label}_step{latest_step}_full.yaml"
save_config(cfg, TEMP_EVAL_CONFIG)
print("Config:", TEMP_EVAL_CONFIG)
print("Eval output:", EVAL_DIR)

In [ ]:
RUN_EVAL = True
if RUN_EVAL:
    run_script("scripts/rlhf_evaluate_policy_suite.py", str(TEMP_EVAL_CONFIG), {})
else:
    print("Skipping evaluation.")

## 9. Qualitative audit and Drive sync

The audit writes repetition diagnostics, reward-model mismatch candidates, strong PPO losses, and report-friendly markdown/CSV files.

In [ ]:
RUN_AUDIT = True
if RUN_AUDIT:
    audit_command = [
        sys.executable,
        "scripts/rlhf_audit_policy_suite.py",
        "--eval-dir", str(EVAL_DIR),
        "--base-label", "base",
        "--sft-label", "sft_trl",
        "--ppo-label", ppo_eval_label,
    ]
    print("$", shlex.join(audit_command))
    subprocess.run(audit_command, cwd=REPO_ROOT, check=True)
    print("Audit:", EVAL_DIR / "qualitative_audit_auto.md")
else:
    print("Skipping qualitative audit.")

copy_tree(EVAL_DIR, DRIVE_RUN_ROOT / EVAL_DIR.name)
if TEMP_EVAL_CONFIG.exists():
    shutil.copy2(TEMP_EVAL_CONFIG, DRIVE_RUN_ROOT / EVAL_DIR.name / TEMP_EVAL_CONFIG.name)

## 10. Lightweight export for local Codex analysis

This mirrors logs, configs, reports, notebooks, CSV/JSON/markdown, and plots into a Drive folder while skipping large model/tensor files. Your local Google Drive app can sync this smaller tree to the MacBook for analysis without pulling multi-GB weights.

In [ ]:
RUN_LIGHTWEIGHT_EXPORT = True
MAX_EXPORT_MB = 50

SKIP_SUFFIXES = {
    ".bin", ".safetensors", ".pt", ".pth", ".ckpt", ".onnx", ".gguf",
    ".zip", ".tar", ".gz", ".bz2", ".xz", ".7z",
}


def export_lightweight_tree(source, destination, max_mb=50):
    source = Path(source)
    destination = Path(destination)
    if not source.exists():
        raise FileNotFoundError(source)
    max_bytes = int(max_mb * 1024 * 1024)
    copied = 0
    skipped = 0
    for path in source.rglob("*"):
        if path.is_dir():
            continue
        rel = path.relative_to(source)
        target = destination / rel
        try:
            size = path.stat().st_size
        except FileNotFoundError:
            continue
        if size > max_bytes or path.suffix.lower() in SKIP_SUFFIXES:
            skipped += 1
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
        copied += 1
    manifest = {
        "source": str(source),
        "destination": str(destination),
        "max_mb": max_mb,
        "copied_files": copied,
        "skipped_files": skipped,
    }
    destination.mkdir(parents=True, exist_ok=True)
    (destination / "_lightweight_clone_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")


if RUN_LIGHTWEIGHT_EXPORT:
    export_lightweight_tree(DRIVE_RUN_ROOT, LIGHTWEIGHT_RUN_ROOT, MAX_EXPORT_MB)
else:
    print("Skipping lightweight export.")

## 11. Portfolio curation/export

Use `rlhf_runs/ckpt100_deep_curation.ipynb` for the final response-explorer JSON export. It reads the full evaluation files, assigns deterministic heuristic labels, marks curated examples, and writes the portfolio JSON. Keeping that curation notebook separate keeps this training control notebook focused.